# 🎬 LIMPIEZA DE DATOS Y MODELO DE CLASIFICACIÓN — NETFLIX CUSTOMER CHURN

**Conjunto de datos:** `netflix_customer_churn.csv` (5.000 clientes, 14 atributos).

**Modelo elegido:** **Regresión Logística (clasificación binaria)**.

**Justificación:** la variable objetivo del conjunto de datos es `churned`, la cual solo toma los valores **0** (el cliente permanece) y **1** (el cliente se da de baja). Se trata por lo tanto de una **variable cualitativa dicotómica**, no de una variable continua, por lo que el método adecuado es un **algoritmo de clasificación** y no una **regresión lineal**. La regresión lineal produciría valores fuera del intervalo [0,1] y violaría sus supuestos (linealidad, homocedasticidad y normalidad de los residuos).

---

### Contenido

1. Declaración de librerías y carga de datos
2. Exploración del conjunto de datos
3. Limpieza de datos
4. Transformación de datos
5. Modelo de clasificación: Regresión Logística
6. Comparación con otros modelos de clasificación
7. Conclusiones

# 1. Declaración de librerías y carga de datos

Se importan las librerías necesarias para la manipulación de datos (**Pandas**, **NumPy**), la visualización (**Matplotlib**, **Seaborn**) y el preprocesamiento y modelado (**Scikit-learn**).

In [ ]:
# Se importan las librerías.
import pandas as pd                 # Manipulación de datos en tablas (dataFrames)
import numpy as np                  # Operaciones numéricas
import matplotlib.pyplot as plt     # Visualización de datos
import seaborn as sns               # Visualización estadística
import warnings                     # Manejo de advertencias

from sklearn import preprocessing                      # Preprocesamiento (codificación y escalado)
from sklearn.model_selection import train_test_split   # División en entrenamiento y prueba
from sklearn.linear_model import LogisticRegression    # Modelo de clasificación
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, roc_curve,
                             classification_report)

# Se define el estilo de las gráficas.
plt.style.use('seaborn-v0_8-whitegrid')

# Se ignoran las alertas.
warnings.filterwarnings('ignore')

# Se muestran todas las columnas de las tablas.
pd.set_option('display.max_columns', None)

print('Librerías cargadas correctamente.')

### Carga del archivo

Ejecuta la siguiente celda y selecciona el archivo **`netflix_customer_churn.csv`** desde tu computadora.

In [ ]:
# Se carga el archivo desde el equipo local (Google Colab).
from google.colab import files
subida = files.upload()

# Se lee el CSV y se almacena en un dataFrame de Pandas.
dataset = pd.read_csv('netflix_customer_churn.csv')

print('Conjunto de datos cargado.')

**Alternativa:** si prefieres cargar el archivo desde Google Drive, comenta la celda anterior y descomenta la siguiente.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# dataset = pd.read_csv('/content/drive/MyDrive/netflix_customer_churn.csv')

# 2. Exploración del conjunto de datos

Antes de modificar la información es necesario **conocerla**. Se revisan sus dimensiones, los tipos de datos de cada columna, la presencia de valores nulos y duplicados, y las estadísticas básicas de cada variable.

In [ ]:
# Se imprimen los primeros cinco renglones del conjunto de datos.
dataset.head()

In [ ]:
# Se imprimen las dimensiones del conjunto de datos (renglones, columnas).
print('El conjunto de datos tiene ' + str(dataset.shape[0]) + ' renglones y ' + str(dataset.shape[1]) + ' columnas.')

La función **`info()`** muestra el nombre de cada columna, el número de valores no nulos (`Non-Null Count`) y el tipo de dato en el que está categorizada cada una (`Dtype`).

In [ ]:
# Se despliega la información general del conjunto de datos.
dataset.info()

La función **`describe()`** despliega las estadísticas básicas de las variables numéricas: número de valores (`count`), promedio (`mean`), desviación estándar (`std`), valor mínimo (`min`), máximo (`max`) y los percentiles 25%, 50% y 75%.

In [ ]:
# Se despliegan las estadísticas de las variables numéricas.
dataset.describe()

In [ ]:
# Se cuentan los valores nulos por columna.
print('VALORES NULOS POR COLUMNA:')
print(dataset.isnull().sum())

# Se cuenta el número de renglones duplicados.
print()
print('El número de renglones duplicados en el conjunto de datos es: ' + str(len(dataset) - len(dataset.drop_duplicates())))

# Se verifica que no existan identificadores de cliente repetidos.
print('El número de identificadores de cliente repetidos es: ' + str(dataset['customer_id'].duplicated().sum()))

### Exploración de la variable objetivo

La variable **`churned`** es la que se desea predecir. Se revisa su distribución para comprobar si las clases están **balanceadas**, lo cual determina qué métricas de evaluación son confiables.

In [ ]:
# Se cuentan los valores de la variable objetivo.
conteo = dataset['churned'].value_counts()
proporcion = dataset['churned'].value_counts(normalize=True)

print('DISTRIBUCIÓN DE LA VARIABLE OBJETIVO (churned):')
print('Clase 0 (Permanece): ' + str(conteo[0]) + ' clientes (' + str(round(proporcion[0]*100, 2)) + '%)')
print('Clase 1 (Abandona):  ' + str(conteo[1]) + ' clientes (' + str(round(proporcion[1]*100, 2)) + '%)')

# Se grafica la distribución.
plt.figure(figsize=(7, 5))
sns.countplot(x='churned', data=dataset, palette=['#4C72B0', '#C44E52'])
plt.title('DISTRIBUCIÓN DE LA VARIABLE OBJETIVO', fontdict={'family': 'DejaVu Sans', 'color': 'black', 'weight': 'bold', 'size': 13}, pad=15)
plt.xlabel('CHURNED (0 = Permanece, 1 = Abandona)', fontdict={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 10}, labelpad=12)
plt.ylabel('NÚMERO DE CLIENTES', fontdict={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 10}, labelpad=12)
plt.suptitle('Fig. 1 Distribución de las clases de la variable objetivo.', fontproperties={'family': 'DejaVu Sans', 'size': 11}, y=-0.02)
plt.show()

Las clases se encuentran prácticamente **balanceadas** (≈50% y ≈50%), lo cual es favorable: no será necesario aplicar técnicas de balanceo como *oversampling* o *undersampling*, y la métrica de **exactitud (accuracy)** será confiable.

### Exploración de las variables cualitativas

Se revisan los valores únicos de cada variable categórica para detectar **inconsistencias** (errores de escritura, mayúsculas mezcladas o categorías duplicadas).

In [ ]:
# Se listan las columnas de tipo cualitativo.
columnas_cualitativas = ['gender', 'subscription_type', 'region', 'device', 'payment_method', 'favorite_genre']

# Se imprimen los valores únicos de cada una.
for columna in columnas_cualitativas:
    print('--- ' + columna.upper() + ' (' + str(dataset[columna].nunique()) + ' categorías) ---')
    print(dataset[columna].value_counts())
    print()

# 3. Limpieza de datos

Con la información obtenida en la exploración, se establece la siguiente **lista de tareas** de limpieza:

* Eliminación de columnas no útiles o redundantes.
* Renombramiento de columnas.
* Remoción de valores nulos.
* Búsqueda y eliminación de datos duplicados.
* Detección y eliminación de valores atípicos (*outliers*).

## 3.1 Eliminación de columnas

Se eliminan tres columnas por las siguientes razones:

* **`customer_id`**: es un **identificador único** sin poder predictivo. Si se incluyera, el modelo intentaría aprender de un dato arbitrario.
* **`monthly_fee`**: es **redundante**, ya que tiene una correspondencia exacta 1 a 1 con `subscription_type` (Basic = 8.99, Standard = 13.99, Premium = 17.99). Mantener ambas introduce información duplicada.
* **`avg_watch_time_per_day`**: presenta **colinealidad** con `watch_hours` (miden el mismo fenómeno en distinta escala). La presencia de variables colineales distorsiona los coeficientes de una regresión logística.

Primero se comprueba la redundancia de `monthly_fee` antes de eliminarla.

In [ ]:
# Se comprueba la correspondencia entre el tipo de suscripción y la tarifa mensual.
print('CORRESPONDENCIA ENTRE subscription_type Y monthly_fee:')
print(dataset.groupby('subscription_type')['monthly_fee'].unique())

In [ ]:
# Se eliminan las columnas no útiles o redundantes.
dataset = dataset.drop(columns=['customer_id', 'monthly_fee', 'avg_watch_time_per_day'])

# Se imprime el conjunto de datos resultante.
print('El conjunto de datos ahora tiene ' + str(dataset.shape[1]) + ' columnas.')
dataset.head()

## 3.2 Renombramiento de columnas

Se cambian los nombres de las columnas al español utilizando la función **`rename()`** de Pandas, con el fin de obtener un análisis y una visualización más cómoda y consistente.

In [ ]:
# Se renombran las columnas.
dataset = dataset.rename(columns={'age': 'Edad',
                                  'gender': 'Género',
                                  'subscription_type': 'Tipo_de_suscripción',
                                  'watch_hours': 'Horas_vistas',
                                  'last_login_days': 'Días_desde_último_acceso',
                                  'region': 'Región',
                                  'device': 'Dispositivo',
                                  'churned': 'Abandono',
                                  'payment_method': 'Método_de_pago',
                                  'number_of_profiles': 'Número_de_perfiles',
                                  'favorite_genre': 'Género_favorito'})

# Se imprime el conjunto de datos.
dataset.head()

## 3.3 Remoción de valores nulos (NaN)

Se eliminan los renglones que contengan valores nulos utilizando la función **`dropna()`**, y se reinicia el índice de la tabla.

In [ ]:
# Se registra el número de renglones antes de la limpieza.
renglones_iniciales = len(dataset)

# Se eliminan los renglones con valores nulos y se resetea el índice.
dataset = dataset.dropna().reset_index(drop=True)

# Se imprime el resultado.
print('Renglones eliminados por valores nulos: ' + str(renglones_iniciales - len(dataset)))
print('Renglones restantes: ' + str(len(dataset)))

## 3.4 Búsqueda y eliminación de datos duplicados

La existencia de renglones duplicados es recurrente en los conjuntos de datos, por lo que debe ser rutinaria la comprobación de su existencia y su posterior eliminación con la función **`drop_duplicates()`**.

In [ ]:
# Se imprime el número de renglones duplicados.
print('El número de renglones duplicados en el conjunto de datos es: ' + str(len(dataset) - len(dataset.drop_duplicates())))

# Se eliminan los renglones repetidos y se resetea el índice.
dataset.drop_duplicates(inplace=True)
dataset = dataset.reset_index(drop=True)

print('Renglones restantes: ' + str(len(dataset)))

## 3.5 Detección y remoción de valores atípicos (outliers)

Para identificar los valores que se desvían considerablemente del comportamiento típico de los datos se utilizará el método estadístico del **rango intercuartílico (IQR)**, una técnica convencional aplicable a distribuciones no gaussianas.

El método define como atípico todo valor que se encuentre fuera del intervalo:

$$[\,Q_1 - 1.5 \cdot IQR\;,\;\; Q_3 + 1.5 \cdot IQR\,]$$

donde $Q_1$ y $Q_3$ son el primer y el tercer cuartil, e $IQR = Q_3 - Q_1$.

Primero se visualiza la dispersión de las dos variables numéricas continuas más relevantes.

In [ ]:
# Se grafica la dispersión de los datos antes de la remoción de atípicos.
plt.figure(figsize=(9, 7))
plt.plot(dataset['Días_desde_último_acceso'], dataset['Horas_vistas'], 'o', markersize=5, alpha=0.5)
plt.title('HORAS VISTAS RESPECTO A DÍAS DESDE EL ÚLTIMO ACCESO', fontdict={'family': 'DejaVu Sans', 'color': 'black', 'weight': 'bold', 'size': 13}, pad=15)
plt.xlabel('DÍAS DESDE EL ÚLTIMO ACCESO', fontdict={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 10}, labelpad=12)
plt.ylabel('HORAS VISTAS', fontdict={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 10}, labelpad=12)
plt.suptitle('Fig. 2 Dispersión de los datos. En la parte superior se pueden notar puntos considerablemente alejados de los demás.', fontproperties={'family': 'DejaVu Sans', 'size': 11}, y=-0.02)
plt.show()

In [ ]:
# Se crea una función que calcula los límites superior e inferior del rango intercuartílico.
def iqrFilter(column):
    percentile25 = dataset[column].quantile(0.25)
    percentile75 = dataset[column].quantile(0.75)
    iqr = percentile75 - percentile25

    upper_limit = percentile75 + 1.5 * iqr
    lower_limit = percentile25 - 1.5 * iqr

    return upper_limit, lower_limit

# Se listan las columnas numéricas a las que se aplicará el filtro.
# Nota: no se incluye la variable objetivo 'Abandono' ni las variables aún cualitativas.
columnas_numericas = ['Edad', 'Horas_vistas', 'Días_desde_último_acceso', 'Número_de_perfiles']

# Se imprimen los límites calculados para cada una.
print('LÍMITES DEL RANGO INTERCUARTÍLICO POR VARIABLE:')
for columna in columnas_numericas:
    superior, inferior = iqrFilter(columna)
    atipicos = len(dataset[(dataset[columna] < inferior) | (dataset[columna] > superior)])
    print(columna + ': [' + str(round(inferior, 2)) + ', ' + str(round(superior, 2)) + '] = ' + str(atipicos) + ' valores atípicos')

Se visualizan los valores detectados como atípicos antes de proceder a su eliminación.

In [ ]:
# Se crea un conjunto de datos temporal para hacer las comparaciones.
temp_dataset = dataset.copy()

# Se detectan y filtran los valores atípicos de todas las columnas numéricas.
for columna in columnas_numericas:
    superior, inferior = iqrFilter(columna)
    temp_dataset = temp_dataset[(temp_dataset[columna] > inferior) & (temp_dataset[columna] < superior)]

# Se genera un conjunto de datos que contiene únicamente los valores atípicos.
out_dataset = pd.concat([dataset, temp_dataset]).drop_duplicates(keep=False)

# Se despliegan las gráficas comparativas.
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(21, 6), sharey=True)

ax1.scatter(dataset['Días_desde_último_acceso'], dataset['Horas_vistas'], alpha=0.5)
ax1.set_title('ANTES DE LA REMOCIÓN', fontdict={'family': 'DejaVu Sans', 'color': 'black', 'weight': 'bold', 'size': 13}, pad=15)
ax1.set_xlabel('DÍAS DESDE EL ÚLTIMO ACCESO', fontdict={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 10}, labelpad=12)
ax1.set_ylabel('HORAS VISTAS', fontdict={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 10}, labelpad=12)

ax2.scatter(dataset['Días_desde_último_acceso'], dataset['Horas_vistas'], alpha=0.5)
ax2.scatter(out_dataset['Días_desde_último_acceso'], out_dataset['Horas_vistas'], c='red', label='Valores atípicos')
ax2.set_title('DATOS ATÍPICOS DETECTADOS', fontdict={'family': 'DejaVu Sans', 'color': 'black', 'weight': 'bold', 'size': 13}, pad=15)
ax2.set_xlabel('DÍAS DESDE EL ÚLTIMO ACCESO', fontdict={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 10}, labelpad=12)
ax2.legend(loc='upper right', prop={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 12}, frameon=True, framealpha=1, facecolor='#dddddd', shadow=True)

ax3.scatter(temp_dataset['Días_desde_último_acceso'], temp_dataset['Horas_vistas'], alpha=0.5)
ax3.set_title('DESPUÉS DE LA REMOCIÓN', fontdict={'family': 'DejaVu Sans', 'color': 'black', 'weight': 'bold', 'size': 13}, pad=15)
ax3.set_xlabel('DÍAS DESDE EL ÚLTIMO ACCESO', fontdict={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 10}, labelpad=12)

plt.suptitle('Fig. 3 Comparación de los datos antes (izquierda), con los atípicos señalados (centro) y después de su remoción (derecha).', fontproperties={'family': 'DejaVu Sans', 'size': 12}, y=-0.02)
plt.show()

print('Valores atípicos detectados: ' + str(len(out_dataset)) + ' (' + str(round(len(out_dataset)/len(dataset)*100, 2)) + '% del total)')

> **⚠ Consideración importante:** los valores atípicos detectados corresponden a clientes con un consumo muy alto de horas. Estos **no son errores de medición**, sino usuarios legítimos de consumo intensivo, y además representan el segmento con la **tasa de abandono más baja**, por lo que aportan información útil al modelo.
>
> Por esta razón se deja el control de esta decisión en la variable `ELIMINAR_ATIPICOS`. Con el valor `True` se aplica el criterio estándar del rango intercuartílico (lo que corresponde al ejercicio de limpieza); con `False` se conservan estos registros. Puedes ejecutar el notebook con ambos valores y comparar las métricas finales del modelo.

In [ ]:
# Se define si se eliminan o no los valores atípicos detectados.
ELIMINAR_ATIPICOS = True

if ELIMINAR_ATIPICOS:
    for columna in columnas_numericas:
        superior, inferior = iqrFilter(columna)
        dataset = dataset.loc[(dataset[columna] > inferior) & (dataset[columna] < superior)]
    dataset = dataset.reset_index(drop=True)
    print('Se eliminaron los valores atípicos.')
else:
    print('Se conservaron los valores atípicos.')

print('El conjunto de datos limpio tiene ' + str(len(dataset)) + ' renglones y ' + str(dataset.shape[1]) + ' columnas.')
dataset.head()

# 4. Transformación de datos

La transformación consiste en expresar los datos en formas que **posibiliten y optimicen** su procesamiento computacional. En esta etapa se llevarán a cabo dos actividades:

* **Codificación** de los datos cualitativos a tipo cuantitativo.
* **Escalado** de los datos numéricos.

## 4.1 Codificación de datos cualitativos

Dado que los datos alimentarán un modelo de aprendizaje automático, **todos los valores deben ser numéricos**. Para lograrlo se utiliza la clase **`LabelEncoder()`** de Scikit-learn, la cual asigna un número entero a cada uno de los posibles valores de la columna.

In [ ]:
# Se verifica cuáles columnas siguen siendo de tipo cualitativo.
dataset.info()

In [ ]:
# Se declara el codificador.
le = preprocessing.LabelEncoder()

# Se listan las columnas cualitativas con sus nuevos nombres.
columnas_a_codificar = ['Género', 'Tipo_de_suscripción', 'Región', 'Dispositivo', 'Método_de_pago', 'Género_favorito']

# Se guarda el diccionario de equivalencias para su posterior interpretación.
equivalencias = {}

# Se aplica el codificador a cada columna con valores no numéricos.
for columna in columnas_a_codificar:
    dataset[columna] = le.fit_transform(dataset[columna])
    equivalencias[columna] = dict(zip(le.classes_, le.transform(le.classes_)))

# Se imprimen las equivalencias generadas.
print('EQUIVALENCIAS DE LA CODIFICACIÓN:')
for columna, mapa in equivalencias.items():
    print()
    print(columna + ':')
    for categoria, codigo in mapa.items():
        print('   ' + str(categoria) + ' = ' + str(codigo))

In [ ]:
# Se imprime el conjunto de datos ya codificado.
dataset.head(10)

> **Nota:** `LabelEncoder()` asigna enteros consecutivos, lo cual introduce un **orden implícito** entre las categorías. Esto es adecuado para `Tipo_de_suscripción` (Basic < Standard < Premium), pero discutible para variables como `Región` o `Género_favorito`, donde no existe una jerarquía natural. La alternativa formal es la codificación **One-Hot** (`pd.get_dummies()`), que crea una columna binaria por categoría. Se mantiene `LabelEncoder()` por consistencia con la metodología vista en clase y por simplicidad del ejercicio.

## 4.2 Escalado de datos

Al revisar las estadísticas del conjunto se observan rangos muy distintos entre las variables (por ejemplo, `Edad` va de 18 a 70 mientras que `Número_de_perfiles` va de 1 a 5). Para evitar que las variables de mayor magnitud dominen el cálculo del modelo, se **escalan** los datos al intervalo [0, 1] utilizando la clase **`MinMaxScaler()`**.

**Importante:** la variable objetivo `Abandono` **no se escala**, ya que debe conservar sus valores originales 0 y 1.

In [ ]:
# Se despliegan las estadísticas antes del escalado.
dataset.describe()

In [ ]:
# Se separa la variable objetivo para que no sea escalada.
objetivo = dataset['Abandono']
caracteristicas = dataset.drop(columns=['Abandono'])

# Se declara la clase que escalará los datos.
scaler = preprocessing.MinMaxScaler()

# Se escalan las características a un intervalo entre 0 y 1.
normalizado = scaler.fit_transform(caracteristicas)

# Se recrea la tabla con los datos escalados.
caracteristicas = pd.DataFrame(normalizado, columns=caracteristicas.columns)

# Se reintegra la variable objetivo al conjunto de datos.
dataset = pd.concat([caracteristicas, objetivo.reset_index(drop=True)], axis=1)

# Se imprime el conjunto de datos preprocesado.
dataset.head()

In [ ]:
# Se despliegan las estadísticas después del escalado.
dataset.describe()

## 4.3 Resultado del preprocesamiento

El conjunto de datos ha quedado completamente **limpio**, **codificado** y **escalado**, listo para alimentar el modelo de machine learning. Se exporta a un archivo CSV para su uso posterior.

In [ ]:
# Se verifica el estado final del conjunto de datos.
print('CONJUNTO DE DATOS PREPROCESADO')
print('Renglones: ' + str(dataset.shape[0]))
print('Columnas:  ' + str(dataset.shape[1]))
print('Valores nulos: ' + str(dataset.isnull().sum().sum()))
print('Renglones duplicados: ' + str(len(dataset) - len(dataset.drop_duplicates())))
print()
dataset.info()

In [ ]:
# Se exporta el conjunto de datos limpio.
dataset.to_csv('netflix_churn_limpio.csv', index=False)

# Se descarga el archivo generado.
files.download('netflix_churn_limpio.csv')

print('Archivo netflix_churn_limpio.csv generado correctamente.')

### Matriz de correlación

Antes de modelar, se visualiza la **correlación** entre las variables para identificar cuáles guardan mayor relación con la variable objetivo.

In [ ]:
# Se calcula la matriz de correlación.
correlacion = dataset.corr()

# Se grafica el mapa de calor.
plt.figure(figsize=(11, 9))
sns.heatmap(correlacion, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('MATRIZ DE CORRELACIÓN', fontdict={'family': 'DejaVu Sans', 'color': 'black', 'weight': 'bold', 'size': 14}, pad=18)
plt.suptitle('Fig. 4 Correlación lineal entre todas las variables del conjunto de datos preprocesado.', fontproperties={'family': 'DejaVu Sans', 'size': 11}, y=-0.02)
plt.show()

# Se imprime la correlación de cada variable con la variable objetivo.
print('CORRELACIÓN CON LA VARIABLE OBJETIVO (Abandono):')
print(correlacion['Abandono'].drop('Abandono').sort_values(ascending=False))

# 5. Modelo de clasificación: Regresión Logística

La **regresión logística** es un algoritmo de clasificación que estima la **probabilidad condicional** de que una observación pertenezca a una clase, mediante la función **sigmoide**:

$$p(y=1\,|\,x,\theta)=\frac{1}{1+e^{-(\theta_{0}+\theta_{1}x_{1}+\theta_{2}x_{2}+\dots+\theta_{n}x_{n})}}$$

Al intervenir varias variables independientes, el modelo a construir es una **regresión logística multivariable**.

## 5.1 Definición de variables y división del conjunto

Se separan las **características de entrada** (X) de las **etiquetas** (y), y se divide el conjunto en **entrenamiento (80%)** y **prueba (20%)**.

In [ ]:
# Se seleccionan todas las columnas excepto la variable objetivo.
X = dataset.drop(columns=['Abandono'])

# Se definen los datos correspondientes a las etiquetas.
y = dataset['Abandono']

print('0: El cliente permanece')
print('1: El cliente abandona')
print()
print('Características de entrada (X): ' + str(X.shape))
print('Etiquetas (y): ' + str(y.shape))
print()
print('Variables utilizadas:')
print(list(X.columns))

In [ ]:
# Se separan los datos de entrenamiento y de prueba.
# random_state=42 asegura que la división sea reproducible.
# stratify=y mantiene la misma proporción de clases en ambos subconjuntos.
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print('Ejemplos de entrenamiento: ' + str(len(X_train)))
print('Ejemplos de prueba:        ' + str(len(X_test)))

## 5.2 Entrenamiento del modelo

Se define el algoritmo y se ajusta a los datos de entrenamiento con el método **`fit()`**.

In [ ]:
# Se define el algoritmo a utilizar.
algoritmo = LogisticRegression(max_iter=1000)

# Se entrena el modelo.
algoritmo.fit(X_train, y_train)

print('Modelo entrenado correctamente.')

In [ ]:
# Se realiza la predicción sobre el conjunto de prueba.
y_pred = algoritmo.predict(X_test)

# Se obtienen también las probabilidades estimadas (necesarias para la curva ROC).
y_prob = algoritmo.predict_proba(X_test)[:, 1]

print('Primeras 15 predicciones:     ' + str(y_pred[:15]))
print('Primeras 15 etiquetas reales: ' + str(y_test.values[:15]))

## 5.3 Evaluación del modelo

### Matriz de confusión

La matriz de confusión permite verificar el rendimiento del modelo mostrando los aciertos y los errores cometidos en cada clase.

In [ ]:
# Se verifica el rendimiento del modelo con la matriz de confusión.
matriz = confusion_matrix(y_test, y_pred)

print('Matriz de confusión:')
print(matriz)

# Se extraen los valores de la matriz.
vn, fp, fn, vp = matriz.ravel()

print()
print('Verdaderos negativos (VN): ' + str(vn) + ' → clientes que permanecen y el modelo acertó.')
print('Falsos positivos (FP):     ' + str(fp) + ' → clientes que permanecen pero el modelo predijo abandono.')
print('Falsos negativos (FN):     ' + str(fn) + ' → clientes que abandonan pero el modelo predijo permanencia.')
print('Verdaderos positivos (VP): ' + str(vp) + ' → clientes que abandonan y el modelo acertó.')
print()
print('En total fueron ' + str(fp + fn) + ' datos en los que el modelo no obtuvo una buena predicción.')

In [ ]:
# Se grafica la matriz de confusión.
plt.figure(figsize=(7, 6))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Permanece (0)', 'Abandona (1)'],
            yticklabels=['Permanece (0)', 'Abandona (1)'],
            annot_kws={'size': 15, 'weight': 'bold'})
plt.title('MATRIZ DE CONFUSIÓN', fontdict={'family': 'DejaVu Sans', 'color': 'black', 'weight': 'bold', 'size': 14}, pad=18)
plt.xlabel('PREDICCIÓN DEL MODELO', fontdict={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 11}, labelpad=12)
plt.ylabel('VALOR REAL', fontdict={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 11}, labelpad=12)
plt.suptitle('Fig. 5 Matriz de confusión del modelo sobre el conjunto de prueba.', fontproperties={'family': 'DejaVu Sans', 'size': 11}, y=-0.02)
plt.show()

### Métricas de rendimiento

Se calculan las cinco métricas estándar para un modelo de clasificación binaria.

In [ ]:
# Cálculo de la exactitud del modelo (accuracy).
exactitud = accuracy_score(y_test, y_pred)
print('Exactitud del modelo (Accuracy):')
print(exactitud)

# Cálculo de la precisión del modelo (precision).
precision = precision_score(y_test, y_pred)
print()
print('Precisión del modelo (Precision):')
print(precision)

# Cálculo de la sensibilidad del modelo (recall).
sensibilidad = recall_score(y_test, y_pred)
print()
print('Sensibilidad del modelo (Recall):')
print(sensibilidad)

# Cálculo del puntaje F1 del modelo.
puntajef1 = f1_score(y_test, y_pred)
print()
print('Puntaje F1 del modelo:')
print(puntajef1)

# Cálculo de la curva ROC - AUC del modelo.
roc_auc = roc_auc_score(y_test, y_prob)
print()
print('Puntaje de la curva ROC - AUC del modelo:')
print(roc_auc)

In [ ]:
# Se imprime el reporte completo de clasificación.
print('REPORTE DE CLASIFICACIÓN:')
print(classification_report(y_test, y_pred, target_names=['Permanece (0)', 'Abandona (1)']))

In [ ]:
# Se calcula y grafica la curva ROC.
fpr, tpr, umbrales = roc_curve(y_test, y_prob)

plt.figure(figsize=(7, 7))
plt.plot(fpr, tpr, color='#C44E52', linewidth=2.5, label='Curva ROC (AUC = ' + str(round(roc_auc, 4)) + ')')
plt.plot([0, 1], [0, 1], color='gray', linewidth=1.5, linestyle='--', label='Clasificador aleatorio (AUC = 0.5)')
plt.title('CURVA ROC', fontdict={'family': 'DejaVu Sans', 'color': 'black', 'weight': 'bold', 'size': 14}, pad=18)
plt.xlabel('TASA DE FALSOS POSITIVOS', fontdict={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 10}, labelpad=12)
plt.ylabel('TASA DE VERDADEROS POSITIVOS', fontdict={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 10}, labelpad=12)
plt.legend(loc='lower right', prop={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 11}, frameon=True, framealpha=1, facecolor='#dddddd', shadow=True)
plt.suptitle('Fig. 6 Curva ROC del modelo. Cuanto más se acerca al vértice superior izquierdo, mejor es su capacidad de discriminación.', fontproperties={'family': 'DejaVu Sans', 'size': 11}, y=-0.02)
plt.show()

## 5.4 Interpretación del modelo

Una de las principales ventajas de la regresión logística frente a otros algoritmos de clasificación es que es **interpretable**: los **coeficientes** $\theta$ indican la dirección y la magnitud con la que cada variable influye en la probabilidad de abandono.

* Un coeficiente **positivo** indica que al aumentar esa variable, **aumenta** la probabilidad de que el cliente abandone.
* Un coeficiente **negativo** indica que al aumentar esa variable, **disminuye** dicha probabilidad.

In [ ]:
# Se extraen los parámetros del modelo.
theta_0 = algoritmo.intercept_[0]   # Término independiente (bias)
theta_n = algoritmo.coef_[0]        # Coeficientes de cada característica

print('Intercepto (theta_0): ' + str(round(theta_0, 4)))
print()

# Se construye una tabla con los coeficientes ordenados por su magnitud.
coeficientes = pd.DataFrame({'Variable': X.columns,
                             'Coeficiente': theta_n})
coeficientes['Magnitud'] = coeficientes['Coeficiente'].abs()
coeficientes = coeficientes.sort_values('Magnitud', ascending=False).reset_index(drop=True)

print('COEFICIENTES DEL MODELO (ordenados por influencia):')
coeficientes[['Variable', 'Coeficiente']]

In [ ]:
# Se grafican los coeficientes del modelo.
orden = coeficientes.sort_values('Coeficiente')
colores = ['#4C72B0' if valor < 0 else '#C44E52' for valor in orden['Coeficiente']]

plt.figure(figsize=(9, 7))
plt.barh(orden['Variable'], orden['Coeficiente'], color=colores)
plt.axvline(0, color='black', linewidth=1)
plt.title('INFLUENCIA DE CADA VARIABLE EN EL ABANDONO', fontdict={'family': 'DejaVu Sans', 'color': 'black', 'weight': 'bold', 'size': 13}, pad=18)
plt.xlabel('COEFICIENTE (θ)', fontdict={'family': 'DejaVu Sans', 'weight': 'bold', 'size': 10}, labelpad=12)
plt.suptitle('Fig. 7 Coeficientes del modelo. En rojo las variables que aumentan la probabilidad de abandono, en azul las que la reducen.', fontproperties={'family': 'DejaVu Sans', 'size': 11}, y=-0.02)
plt.show()

## 5.5 Utilizando el modelo

Se utiliza el modelo entrenado para estimar la probabilidad de abandono de un cliente concreto.

In [ ]:
# Se toma un cliente del conjunto de prueba como ejemplo.
indice_ejemplo = 0
cliente = X_test.iloc[[indice_ejemplo]]

# Se calcula la probabilidad y la clase estimada.
probabilidad = algoritmo.predict_proba(cliente)[0][1]
clase = algoritmo.predict(cliente)[0]

print('DATOS DEL CLIENTE (valores escalados entre 0 y 1):')
print(cliente.T)
print()
print('La probabilidad de que el cliente abandone es: ' + str(round(probabilidad, 4)))
print('La clase estimada para el cliente es: ' + str(clase) + ' (' + ('Abandona' if clase == 1 else 'Permanece') + ')')
print('La clase real del cliente es: ' + str(y_test.iloc[indice_ejemplo]))

# 6. Comparación con otros modelos de clasificación

Se prueban otros modelos vistos en clase sobre el mismo conjunto de datos, evaluados con **cinco métricas** cada uno.

**Modelos comparados:** Regresión Logística, Árbol de Decisión, Random Forest y Máquina de Soporte Vectorial (SVM).

> Los modelos de **regresión lineal, múltiple y polinomial** no se incluyen porque predicen valores continuos, y aquí la variable objetivo es binaria (0 o 1). Son modelos de regresión, no de clasificación.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Se definen los modelos a comparar.
modelos = {
    'Regresión Logística': LogisticRegression(max_iter=1000),
    'Árbol de Decisión': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(random_state=42)
}

# Se entrena cada uno y se calculan sus métricas.
resultados = []
for nombre, m in modelos.items():
    m.fit(X_train, y_train)
    pred = m.predict(X_test)
    prob = m.predict_proba(X_test)[:, 1] if hasattr(m, 'predict_proba') else m.decision_function(X_test)
    resultados.append({
        'Modelo': nombre,
        'Exactitud': accuracy_score(y_test, pred),
        'Precisión': precision_score(y_test, pred),
        'Sensibilidad': recall_score(y_test, pred),
        'F1': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, prob)
    })

tabla = pd.DataFrame(resultados).set_index('Modelo').round(4)
tabla

In [ ]:
# Se grafica la comparación.
tabla.plot(kind='bar', figsize=(11, 5), width=0.8)
plt.title('Comparación de modelos de clasificación')
plt.ylabel('Puntaje')
plt.ylim(0, 1.25)
plt.xticks(rotation=10)
plt.legend(loc='upper center', ncol=5, fontsize=9)
plt.grid(axis='y', alpha=0.3)
plt.show()

print('Mejor modelo según F1     :', tabla['F1'].idxmax(), round(tabla['F1'].max(), 4))
print('Mejor modelo según ROC-AUC:', tabla['ROC-AUC'].idxmax(), round(tabla['ROC-AUC'].max(), 4))

In [ ]:
# Matriz de confusión de cada modelo.
fig, ejes = plt.subplots(1, 4, figsize=(18, 4))

for eje, (nombre, m) in zip(ejes, modelos.items()):
    sns.heatmap(confusion_matrix(y_test, m.predict(X_test)), annot=True, fmt='d',
                cmap='Blues', cbar=False, ax=eje,
                xticklabels=['Perm.', 'Aband.'], yticklabels=['Perm.', 'Aband.'])
    eje.set_title(nombre)
    eje.set_xlabel('Predicción')

ejes[0].set_ylabel('Valor real')
plt.tight_layout()
plt.show()

# 7. Conclusiones

**Sobre la limpieza y organización de los datos:**

1. El conjunto de datos original contaba con **5.000 registros y 14 atributos**, sin valores nulos ni registros duplicados, lo cual indica una buena calidad de origen.
2. Se eliminaron **3 columnas**: `customer_id` por ser un identificador sin poder predictivo, `monthly_fee` por ser redundante con `subscription_type`, y `avg_watch_time_per_day` por presentar colinealidad con `watch_hours`.
3. Se renombraron las columnas al español, se aplicó el filtro de **rango intercuartílico** para la remoción de valores atípicos, se codificaron las **6 variables cualitativas** con `LabelEncoder()` y se escalaron todas las características al intervalo [0, 1] con `MinMaxScaler()`.
4. El resultado es un conjunto de datos completamente numérico, escalado y sin inconsistencias, exportado como **`netflix_churn_limpio.csv`**.

**Sobre la elección del modelo:**

5. El modelo apropiado es la **regresión logística (clasificación binaria)**, ya que la variable objetivo `churned` es **dicotómica**. La regresión lineal fue descartada porque requiere una variable de salida continua y generaría predicciones fuera del intervalo [0, 1], imposibles de interpretar como probabilidades.
6. Las clases del conjunto de datos están **balanceadas** (≈50% / 50%), por lo que la métrica de exactitud es confiable; aun así, la evaluación se complementó con precisión, sensibilidad, puntaje F1 y curva ROC-AUC.
7. El análisis de los coeficientes muestra que las variables con mayor influencia en el abandono son las **horas vistas** (a mayor consumo, menor probabilidad de abandono) y los **días desde el último acceso** (a mayor inactividad, mayor probabilidad de abandono).

**Posibles mejoras:**

8. Probar la codificación **One-Hot** en lugar de `LabelEncoder()` para las variables sin orden natural (`Región`, `Género_favorito`, `Método_de_pago`).
9. Comparar el desempeño contra algoritmos no lineales como **Árbol de Decisión** o **Random Forest**, que podrían capturar mejor los cortes no lineales observados en `Horas_vistas`.